### Test WhisperX installation ###

In [1]:
import sys
import os
from pathlib import Path
import logging
import gc
import glob

# PyTorch
import torch

# WhisperX
import whisperx

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

%load_ext autoreload
%autoreload 2
import audiotranscription
print(f'Authors: {audiotranscription.__authors__}')
print(f'Project version: {audiotranscription.__version__}')
print(f'Python version:  {sys.version}')
print(f'PyTorch version: {torch.__version__} with CUDA: {torch.cuda.is_available()}')

Authors: The Core for Computational Biomedicine at Harvard Medical School
https://dbmi.hms.harvard.edu/about-dbmi/core-computational-biomedicine
Project version: v0.0.1
Python version:  3.12.10 (main, May 22 2025, 01:29:12) [GCC 12.2.0]
PyTorch version: 2.8.0+cu128 with CUDA: True


In [2]:
# Parameters and files
device = "cuda"
audio_file = "audio.mp3"
batch_size = 16 # reduce if low on GPU mem
compute_type = "float16" # change to "int8" if low on GPU mem (may reduce accuracy)
whisperx_model = 'small.en'

# Directories
project_name = 'proj1'
input_dir = os.path.join('/app', 'data', 'input', project_name)
output_dir = os.path.join('/app', 'data', 'output', f'{project_name}_output')
Path(output_dir).mkdir(parents=True, exist_ok=True)
print(input_dir)
print(output_dir)

/app/data/input/proj1
/app/data/output/proj1_output


In [3]:
# Select a file
file_list = glob.glob(os.path.join(input_dir, '*.m4a'))
file = file_list[-1]
print(file)

/app/data/input/proj1/compose.m4a


In [4]:
# Load the model
model = whisperx.load_model(whisperx_model, device, compute_type=compute_type)

vocabulary.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/484M [00:00<?, ?B/s]

2026-03-09 18:51:49 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../usr/local/lib/python3.12/site-packages/whisperx/assets/pytorch_model.bin`


In [ ]:
audio = whisperx.load_audio(audio_file)

In [ ]:
result = model.transcribe(audio, batch_size=batch_size)

In [ ]:



print(result["segments"]) # before alignment

# delete model if low on GPU resources
# import gc; import torch; gc.collect(); torch.cuda.empty_cache(); del model

# 2. Align whisper output
model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

print(result["segments"]) # after alignment

# delete model if low on GPU resources
# import gc; import torch; gc.collect(); torch.cuda.empty_cache(); del model_a

# 3. Assign speaker labels
diarize_model = DiarizationPipeline(token=YOUR_HF_TOKEN, device=device)

# add min/max number of speakers if known
diarize_segments = diarize_model(audio)
# diarize_model(audio, min_speakers=min_speakers, max_speakers=max_speakers)

result = whisperx.assign_word_speakers(diarize_segments, result)
print(diarize_segments)
print(result["segments"]) # segments are now assigned speaker IDs